In [ ]:
# 基于Procedural-modeling-applied-to-the-3D-city-mod_2021_Virtual-Reality---Intell文章

In [1]:
import xml.etree.ElementTree as ET
import pandas as pd
import geopandas as gpd

In [2]:
# 读取 GML 文件（如果已转换成 GML 或 Shapefile）
gdf = gpd.read_file(r"C:\Users\86125\Downloads\vic-lgovt-com-com-clue-business-establishments-per-clue-industry-block-2014-na.xml")


In [3]:
gdf

,gml_id,accommodation,admin_and_support_services,agriculture_and_mining,arts_and_recreation_services,block_id,business_services,census_year,clue_small_area,construction,...,other_services,public_administration_and_safety,real_estate_services,rental_and_hiring_services,retail_trade,total_establishments_in_block,transport_postal_and_storage,wholesale_trade,clue_area,geometry
0,com_clue_business_establishments_per_clue_indu...,0,0,0,1,662,0,2014,East Melbourne,0,...,0,0,0,0,0,2,1,0,East Melbourne,"MULTIPOLYGON (((144.98996 -37.81761, 144.98881..."
1,com_clue_business_establishments_per_clue_indu...,0,0,0,0,1112,0,2014,Docklands,0,...,1,0,0,0,1,3,0,0,Docklands,"MULTIPOLYGON (((144.94792 -37.82337, 144.9477 ..."
2,com_clue_business_establishments_per_clue_indu...,0,3,0,6,6,0,2014,Melbourne (CBD),0,...,2,2,2,1,6,45,6,0,Melbourne (CBD),"MULTIPOLYGON (((144.9714 -37.81909, 144.96825 ..."
3,com_clue_business_establishments_per_clue_indu...,0,0,0,1,927,0,2014,Parkville,0,...,0,0,0,0,0,2,0,0,Parkville,"MULTIPOLYGON (((144.94262 -37.78663, 144.94199..."
4,com_clue_business_establishments_per_clue_indu...,0,0,0,0,926,0,2014,Parkville,0,...,0,0,0,0,0,0,0,0,Parkville,"MULTIPOLYGON (((144.94214 -37.78757, 144.941 -..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
601,com_clue_business_establishments_per_clue_indu...,0,0,0,0,2392,0,2014,North Melbourne,0,...,3,0,0,0,3,12,1,0,North Melbourne,"MULTIPOLYGON (((144.94015 -37.79559, 144.9401 ..."
602,com_clue_business_establishments_per_clue_indu...,1,0,0,0,2391,0,2014,North Melbourne,0,...,0,0,0,0,0,3,0,0,North Melbourne,"MULTIPOLYGON (((144.9405 -37.79384, 144.94051 ..."
603,com_clue_business_establishments_per_clue_indu...,0,1,0,0,2383,1,2014,North Melbourne,4,...,1,2,0,0,1,17,0,1,North Melbourne,"MULTIPOLYGON (((144.94001 -37.78917, 144.94001..."
604,com_clue_business_establishments_per_clue_indu...,0,0,0,0,1111,0,2014,West Melbourne (Industrial),0,...,0,0,0,0,0,3,3,0,West Melbourne (Industrial),"MULTIPOLYGON (((144.93178 -37.81799, 144.93163..."


In [4]:
# 读取 XML/GML 文件
tree = ET.parse(r"C:\Users\86125\Downloads\vic-lgovt-com-com-clue-business-establishments-per-clue-industry-block-2014-na.xml")
root = tree.getroot()

In [5]:
tree

In [6]:
# 命名空间定义（需根据文件设置）
ns = {
    'aurin': 'http://datasource-VIC_LGovt_COM-UoM_AURIN_DB',
    'gml': 'http://www.opengis.net/gml/3.2',
    'wfs': 'http://www.opengis.net/wfs/2.0'
}

# 提取所有记录
data = []
for member in root.findall("wfs:member", ns):
    feature = member.find("aurin:com_clue_business_establishments_per_clue_industry_block_2014", ns)
    record = {}
    for field in feature:
        tag = field.tag.split("}")[1]
        if field.text is not None and tag != 'wkb_geometry':
            try:
                record[tag] = int(field.text)
            except ValueError:
                record[tag] = field.text  # e.g., area name
    data.append(record)

# 转为 DataFrame
df = pd.DataFrame(data)


In [7]:
category_columns = [
    'accommodation', 'admin_and_support_services', 'agriculture_and_mining',
    'arts_and_recreation_services', 'business_services', 'construction',
    'education_and_training', 'electricity_gas_water_and_waste_services',
    'finance_and_insurance', 'food_and_beverage_services',
    'health_care_and_social_assistance', 'information_media_and_telecommunications',
    'manufacturing', 'other_services', 'public_administration_and_safety',
    'real_estate_services', 'rental_and_hiring_services', 'retail_trade',
    'transport_postal_and_storage', 'wholesale_trade'
]

df['calculated_total'] = df[category_columns].sum(axis=1)
df['correct'] = df['calculated_total'] == df['total_establishments_in_block']

# 输出错误记录数
print(f"属性数据不一致的记录数: {(~df['correct']).sum()}")


属性数据不一致的记录数: 0


In [8]:

# 几何有效性验证
invalid_geometries = gdf[~gdf.is_valid]
print(f"几何无效记录数（存在自交或未闭合）: {len(invalid_geometries)}")

# 重叠验证
from shapely.ops import unary_union

union_geom = unary_union(gdf.geometry)
if union_geom.is_valid:
    print("所有多边形空间关系无重叠，拓扑一致。")
else:
    print("存在重叠或拓扑错误。")


几何无效记录数（存在自交或未闭合）: 0
所有多边形空间关系无重叠，拓扑一致。


In [9]:
import networkx as nx
from shapely.geometry import Polygon

# 建立邻接图
G = nx.Graph()
for i, geom1 in enumerate(gdf.geometry):
    for j, geom2 in enumerate(gdf.geometry):
        if i < j and geom1.touches(geom2):
            G.add_edge(i, j)

print(f"孤立区域数量: {nx.number_connected_components(G)}")


孤立区域数量: 1
